# GRASP — RAGTruth confirmation + CCP timing on the same GPU

Self-contained. Extracts GRASP features (top-4 heads/layer) and top-24 token
probabilities for RAGTruth's Mistral-7B-Instruct responses, then times MUCH's CCP
and real generation on this GPU. No evaluation happens here: the frozen GRASP-v2
is scored locally by `scripts/16_ragtruth_eval.py`, whose rules were fixed first.

In [ ]:
import base64, io, json, os, subprocess, sys, time, zipfile
from pathlib import Path

WORK    = Path('/kaggle/working')
PROJECT = WORK / 'ai_project'
RESULTS = WORK / 'results'
MUCH_PARENT = Path('/kaggle/tmp')          # big data stays out of the saved output
MUCH    = MUCH_PARENT / 'much'
for p in (PROJECT, RESULTS, MUCH_PARENT):
    p.mkdir(parents=True, exist_ok=True)

CODE_ZIP = '...'
zipfile.ZipFile(io.BytesIO(base64.b64decode(CODE_ZIP))).extractall(PROJECT)
sys.path.insert(0, str(PROJECT / 'src'))

LOG = RESULTS / 'run_log.txt'

def sh(cmd, cwd=None, check=True):
    """Run a command, stream its output to the cell and to run_log.txt."""
    print('$', cmd, flush=True)
    t0 = time.time()
    with open(LOG, 'a', encoding='utf-8') as log:
        log.write(f'\n$ {cmd}\n')
        proc = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end='', flush=True)
            log.write(line)
        rc = proc.wait()
        log.write(f'[exit {rc}, {time.time() - t0:.0f}s]\n')
    if check and rc != 0:
        raise RuntimeError(f'step failed (exit {rc}): {cmd}')
    return rc

def disk(tag=''):
    sh(f'df -h {WORK} {MUCH_PARENT} / | sed "1!s/^/  /"; echo "  [{tag}]"; du -sh {RESULTS} {MUCH_PARENT} ~/.cache/huggingface 2>/dev/null', check=False)

sh('nvidia-smi -L')
print('python', sys.version.split()[0])
disk('start')

In [ ]:
# HuggingFace login. Uses the HF_TOKEN Kaggle secret if one is attached; otherwise
# falls back to HuggingFace's device-code login (open a URL, enter a short code),
# the same flow Colab's login() shows. Implemented with plain requests so it does
# not depend on the preinstalled huggingface_hub version. The token is never printed.
import requests

def device_code_login(timeout_s=1800):
    hf, client = 'https://huggingface.co', '26be6b09-91c5-47da-9861-d2d2bb7a7e36'
    info = requests.post(f'{hf}/oauth/device', data={'client_id': client}, timeout=30)
    info.raise_for_status(); info = info.json()
    url = info.get('verification_uri_complete') or info['verification_uri']
    print('=' * 60)
    print(f"HF-DEVICE-LOGIN url={url} code={info['user_code']}")
    print('Open the URL, sign in as your HF user, enter the code, authorize.')
    print('=' * 60, flush=True)
    interval = int(info.get('interval', 5))
    deadline = time.time() + min(int(info.get('expires_in', 900)), timeout_s)
    while time.time() < deadline:
        time.sleep(interval)
        try:
            r = requests.post(f'{hf}/oauth/token', timeout=30, data={
                'grant_type': 'urn:ietf:params:oauth:grant-type:device_code',
                'device_code': info['device_code'], 'client_id': client}).json()
        except Exception:
            continue
        if 'access_token' in r:
            print('authorized; token valid for', r.get('expires_in', '?'), 's', flush=True)
            return r['access_token']
        err = r.get('error')
        if err == 'slow_down':
            interval += 5
        elif err == 'expired_token':
            return None                      # caller asks for a fresh code
        elif err not in (None, 'authorization_pending'):
            raise SystemExit(f'HF device login failed: {err}')
    return None

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
    print('using the HF_TOKEN Kaggle secret')
except Exception:
    print('no HF_TOKEN secret attached; using device-code login')
t_end = time.time() + 2 * 3600            # keep offering fresh codes for 2 hours
while not token and time.time() < t_end:
    token = device_code_login()
    if not token:
        print('code expired without authorization; requesting a new one', flush=True)
if not token:
    raise SystemExit('HF device login was never authorized.')
os.environ['HF_TOKEN'] = token          # inherited by every subprocess below
tok_file = Path.home() / '.cache' / 'huggingface' / 'token'
tok_file.parent.mkdir(parents=True, exist_ok=True)
tok_file.write_text(token)
who = requests.get('https://huggingface.co/api/whoami-v2',
                   headers={'Authorization': f'Bearer {token}'}, timeout=30).json()
print('HF login ok as', who.get('name'))

## 1. MUCH code and data (for CCP and the claim segmenter)

In [ ]:
sh(f'{sys.executable} {PROJECT}/scripts/01_setup_much.py --root {MUCH_PARENT}')
sh(f'{sys.executable} -m src import-much', cwd=MUCH)
sh(f'{sys.executable} {PROJECT}/src/tests/test_grasp.py')

## 2. RAGTruth extraction, CCP and generation timing

In [ ]:
RT = RESULTS / 'ragtruth'
sh(f'{sys.executable} {PROJECT}/scripts/15_ragtruth_extract.py --much-root {MUCH} '
   f'--data-dir {MUCH_PARENT}/ragtruth --out {RT} --n-train 1500 --calib 100 --top-k 4 '
   f'--gen-n 15 --ccp-test-n 450 --ccp-much-n 30')
print(json.loads((RT / 'timing.json').read_text()))

In [ ]:
import shutil
shutil.make_archive(str(WORK / 'grasp_ragtruth'), 'zip', RESULTS)
print('done')